In [1]:
# this code is based on Andrej Karpathy's "Let's build GPT from scratch" Youtube lecture
  # (another dataset was used here - see alllines_processed.txt)
# https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=7

In [2]:
!wget https://raw.githubusercontent.com/ruxandrailiescu/mini-gpt/main/data/alllines_processed.txt

--2026-01-12 11:46:00--  https://raw.githubusercontent.com/ruxandrailiescu/mini-gpt/main/data/alllines_processed.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4871678 (4.6M) [text/plain]
Saving to: ‘alllines_processed.txt’

alllines_processed. 100%[===================>]   4.65M  --.-KB/s    in 0.03s   

2026-01-12 11:46:00 (163 MB/s) - ‘alllines_processed.txt’ saved [4871678/4871678]



In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
from transformers import GPT2Tokenizer
import inspect
import os

torch.manual_seed(5454)
device = 'cuda'
batch_size = 32
context_length = 512
max_iters = 30000
eval_interval = 1000
eval_iters = 200
n_embed = 384
n_head = 6
n_layer = 6
dropout = 0.2

In [4]:
out_dir = 'out'
always_save_checkpoint = True
learning_rate = 6e-4
weight_decay = 1e-1
beta1 = 0.9
beta2 = 0.95
decay_lr = True
warmup_iters = 1000
lr_decay_iters = 30000
min_lr = 6e-5

In [5]:
with open('alllines_processed.txt', 'r') as f:
  text = f.read()
print(text[:1000])


KING HENRY IV: So shaken as we are, so wan with care,
		Find we a time for frighted peace to pant,
		And breathe short-winded accents of new broils
		To be commenced in strands afar remote.
		No more the thirsty entrance of this soil
		Shall daub her lips with her own children's blood,
		Nor more shall trenching war channel her fields,
		Nor bruise her flowerets with the armed hoofs
		Of hostile paces: those opposed eyes,
		Which, like the meteors of a troubled heaven,
		All of one nature, of one substance bred,
		Did lately meet in the intestine shock
		And furious close of civil butchery
		Shall now, in mutual well-beseeming ranks,
		March all one way and be no more opposed
		Against acquaintance, kindred and allies:
		The edge of war, like an ill-sheathed knife,
		No more shall cut his master. Therefore, friends,
		As far as to the sepulchre of Christ,
		Whose soldier now, under whose blessed cross
		We are impressed and engaged to fight,
		Forthwith a power of English shall we lev

In [6]:
# subword tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2')
sample = tokenizer('hello worlddd')
print(sample)

data = torch.tensor(tokenizer(text)['input_ids'], dtype=torch.long)
print(data[:100])
print(data.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

{'input_ids': [31373, 995, 1860], 'attention_mask': [1, 1, 1]}


Token indices sequence length is longer than the specified maximum sequence length for this model (1574059 > 1024). Running this sequence through the model will result in indexing errors


tensor([  198, 37286,   367,  1677, 18276,  8363,    25,  1406, 27821,   355,
          356,   389,    11,   523,   266,   272,   351,  1337,    11,   198,
          197,   197, 16742,   356,   257,   640,   329, 12773,   276,  4167,
          284, 15857,    11,   198,   197,   197,  1870, 18044,  1790,    12,
         7972,   276, 39271,   286,   649,  1379,  4487,   198,   197,   197,
         2514,   307, 32400,   287, 36593, 44246,  6569,    13,   198,   197,
          197,  2949,   517,   262, 47124, 10384,   286,   428,  9260,   198,
          197,   197,  2484,   439, 12379,   549,   607, 11914,   351,   607,
          898,  1751,   338,  2910,    11,   198,   197,   197, 21991,   517,
         2236, 35091,   278,  1175,  6518,   607,  7032,    11,   198,   197])
torch.Size([1574059])


In [7]:
# train-test split (90-10)
n = int(0.9*len(data))
train = data[:n+1]
test = data[n+1:]

print(train.shape)
print(test.shape)
print(tokenizer.decode(train[-10:].tolist()))   # check for data leakage
print(tokenizer.decode(test[:10].tolist()))

torch.Size([1416654])
torch.Size([157405])
 room in Priam's palace.
		
Enter PRIAM, HECTOR, TR


In [8]:
vocab_size = len(tokenizer)
print(vocab_size)

50257


In [9]:
# load a batch of data
def get_batch(split):
  data = train if split == 'train' else test
  ix = torch.randint(len(data) - context_length, (batch_size,))
  x = torch.stack([data[i:i+context_length] for i in ix])
  y = torch.stack([data[i+1:i+context_length+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

In [17]:
class LinearLayer(nn.Module):
  """ affine linear transformation """

  def __init__(self, in_features, out_features, bias=True):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.weight = nn.Parameter(torch.randn(out_features, in_features))

    if bias:
      self.bias = nn.Parameter(torch.zeros(out_features))
    else:
      self.register_parameter('bias', None)

    self.reset_parameters()

  def reset_parameters(self):
    sqrt_k = 1. / math.sqrt(self.in_features)
    self.weight.data.uniform_(-sqrt_k, sqrt_k)
    if self.bias is not None:
      self.bias.data.uniform_(-sqrt_k, sqrt_k)

  def forward(self, x):
    x = x @ self.weight.T
    if self.bias is not None:
      x = x + self.bias
    return x


class Head(nn.Module):
  """ single head of self-attention """

  def __init__(self, head_size, n_embed, context_length, dropout):
    super().__init__()
    self.key = LinearLayer(n_embed, head_size, bias=False)  # using the custom linear layer
    self.query = LinearLayer(n_embed, head_size, bias=False)
    self.value = LinearLayer(n_embed, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(context_length, context_length)))  # attention mask
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # input: (batch, time-step, embed)
    # output: (batch, time-step, head_size)
    b,t,e = x.shape
    k = self.key(x) # (b, t, hs)
    q = self.query(x) # (b, t, hs)

    # scaled dot-attention
    w = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (b, t, hs) @ (b, t, hs) --> (b, t, t)
    w = w.masked_fill(self.tril[:t, :t] == 0, float('-inf'))
    w = F.softmax(w, dim=-1)
    w = self.dropout(w)

    v = self.value(x) # (b, t, hs)
    out = w @ v # (b, t, t) @ (b, t, hs) --> (b, t, hs)
    return out


class MultiHeadAttention(nn.Module):
  """ multiple self-attention heads in parallel """

  def __init__(self, num_heads, head_size, n_embed, context_length, dropout):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size, n_embed, context_length, dropout) for _ in range(num_heads)])
    self.proj = LinearLayer(head_size * num_heads, n_embed)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    out = torch.cat([h(x) for h in self.heads], dim=-1)
    out = self.dropout(self.proj(out))
    return out


class FeedForward(nn.Module):
  """ feed-forward network with 2 layers and a non-linear activation in between """

  def __init__(self, n_embed, dropout):
    super().__init__()
    self.net = nn.Sequential(
        LinearLayer(n_embed, n_embed*4),
        nn.ReLU(),
        LinearLayer(n_embed*4, n_embed),
        nn.Dropout(dropout),
    )

  def forward(self, x):
    return self.net(x)


class LayerNormalization(nn.Module):
  """ computes statistics across a single sample (features dimension) """

  def __init__(self, dim, eps=1e-5):
    super().__init__()
    self.eps = eps
    self.gamma = nn.Parameter(torch.ones(dim))
    self.beta = nn.Parameter(torch.zeros(dim))

  def forward(self, x):
    xmean = x.mean(-1, keepdim=True)
    xvar = x.var(-1, keepdim=True)
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    return self.out


class Block(nn.Module):
  """ transformer block """

  def __init__(self, n_embed, n_head, dropout, context_length):
    super().__init__()
    head_size = n_embed // n_head
    self.mha = MultiHeadAttention(n_head, head_size, n_embed, context_length, dropout)
    self.ffn = FeedForward(n_embed, dropout)
    self.ln1 = LayerNormalization(n_embed)
    self.ln2 = LayerNormalization(n_embed)

  def forward(self, x):
    x = x + self.mha(self.ln1(x))
    x = x + self.ffn(self.ln2(x))
    return x


class MiniGPT(nn.Module):
  """ entire decoder applying the transformer block times n_layer"""

  def __init__(self, n_layer, n_head, n_embed, context_length, vocab_size, dropout):
    super().__init__()
    self.context_length = context_length
    self.token_embed_table = nn.Embedding(vocab_size, n_embed)
    self.position_embed_table = nn.Embedding(context_length, n_embed)
    self.blocks = nn.Sequential(*[Block(n_embed, n_head, dropout, context_length) for _ in range(n_layer)])
    self.layern = LayerNormalization(n_embed)
    self.lin = LinearLayer(n_embed, vocab_size)

  def forward(self, idx, targets=None):
    b, t = idx.shape
    tok_emb = self.token_embed_table(idx) # (b, t, e)
    pos_emb = self.position_embed_table(torch.arange(t, device=device)) # (t, e)
    x = tok_emb + pos_emb # (b, t, e)
    x = self.blocks(x) # (b, t, e)
    x = self.layern(x)
    logits = self.lin(x) # (b, t, vocab_size)

    if targets is None:
        loss = None
    else:
        b, t, vs = logits.shape
        logits = logits.view(b*t, vs)
        targets = targets.view(b*t)
        loss = F.cross_entropy(logits, targets)

    return logits, loss

  def generate(self, idx, max_new_tokens, temperature=1.0):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -self.context_length:]
        logits, loss = self(idx_cond)
        logits = logits[:, -1, :] / temperature # (b, vs)
        probs = F.softmax(logits, dim=-1) # (b, vs)
        idx_next = torch.multinomial(probs, num_samples=1) # (b, 1)
        idx = torch.cat((idx, idx_next), dim=1) # (b, t+1)
    return idx

  def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
    param_dict = {pn: p for pn, p in self.named_parameters()}
    param_dict = {pn: p for pn, p in param_dict.items() if p.requires_grad}
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    num_decay_params = sum(p.numel() for p in decay_params)
    num_nodecay_params = sum(p.numel() for p in nodecay_params)
    print(f"num decayed parameter tensors: {len(decay_params)}, with {num_decay_params:,} parameters")
    print(f"num non-decayed parameter tensors: {len(nodecay_params)}, with {num_nodecay_params:,} parameters")
    fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
    use_fused = fused_available and device_type == 'cuda'
    extra_args = dict(fused=True) if use_fused else dict()
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)
    print(f"using fused AdamW: {use_fused}")

    return optimizer

In [11]:
iter_num = 0
best_val_loss = 1e9
checkpoint = None

In [12]:
model_args = dict(n_layer=n_layer, n_head=n_head, n_embed=n_embed, context_length=context_length,
                  vocab_size=vocab_size, dropout=dropout)
model = MiniGPT()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')
optimizer = model.configure_optimizers(weight_decay, learning_rate, (beta1, beta2), device)

49.484881 M parameters
num decayed parameter tensors: 129, with 49,410,816 parameters
num non-decayed parameter tensors: 45, with 74,065 parameters
using fused AdamW: True


In [13]:
# calculate loss and perplexity
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        mean_loss = losses.mean().item()
        perplexity = math.exp(mean_loss)
        out[split] = {
            'loss': mean_loss,
            'perplexity': perplexity
        }
    model.train()
    return out

In [14]:
# learning rate decay scheduler (cosine with warmup)
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

In [15]:
results = []

while True:
  lr = get_lr(iter_num) if decay_lr else learning_rate
  for param_group in optimizer.param_groups:
    param_group['lr'] = lr

  if iter_num % eval_interval == 0:
    losses = estimate_loss()
    results.append(losses)
    print(f"step {iter_num}: train loss {losses['train']['loss']:.4f}, train perplexity {losses['train']['perplexity']:.4f} \
    val loss {losses['val']['loss']:.4f}, val perplexity {losses['val']['perplexity']:.4f}")
    if losses['val']['loss'] < best_val_loss:
      best_val_loss = losses['val']['loss']
      if iter_num > 0:
        checkpoint = {
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'model_args': model_args,
            'iter_num': iter_num,
            'best_val_loss': best_val_loss,
        }
        print(f"saving checkpoint to {out_dir}")
        if not os.path.exists(out_dir):
          os.makedirs(out_dir)
        torch.save(checkpoint, os.path.join(out_dir, 'ckpt_large.pt'))

  xb, yb = get_batch('train')
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

  iter_num += 1

  if iter_num > max_iters:
    break

step 0: train loss 10.9942, train perplexity 59525.9887     val loss 10.9983, val perplexity 59771.4491
step 1000: train loss 3.8149, train perplexity 45.3727     val loss 4.5296, val perplexity 92.7248
saving checkpoint to out
step 2000: train loss 3.0599, train perplexity 21.3262     val loss 4.2611, val perplexity 70.8912
saving checkpoint to out
step 3000: train loss 2.6028, train perplexity 13.5012     val loss 4.3226, val perplexity 75.3869
step 4000: train loss 2.2187, train perplexity 9.1949     val loss 4.4451, val perplexity 85.2091
step 5000: train loss 1.8921, train perplexity 6.6336     val loss 4.6258, val perplexity 102.0856
step 6000: train loss 1.6208, train perplexity 5.0571     val loss 4.8224, val perplexity 124.2673
step 7000: train loss 1.3954, train perplexity 4.0365     val loss 4.9673, val perplexity 143.6405
step 8000: train loss 1.2149, train perplexity 3.3701     val loss 5.1117, val perplexity 165.9507
step 9000: train loss 1.0672, train perplexity 2.9071  

In [18]:
# load checkpoint
ckpt = torch.load('out/ckpt_large.pt', map_location=device)
ckpt_model_args = ckpt['model_args']
m = MiniGPT(**ckpt_model_args)
state_dict = ckpt['model']
m.load_state_dict(state_dict)
m.to(device)
m.eval()

MiniGPT(
  (token_embed_table): Embedding(50257, 384)
  (position_embed_table): Embedding(512, 384)
  (blocks): Sequential(
    (0): Block(
      (mha): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): LinearLayer()
            (query): LinearLayer()
            (value): LinearLayer()
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): LinearLayer()
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): LinearLayer()
          (1): ReLU()
          (2): LinearLayer()
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNormalization()
      (ln2): LayerNormalization()
    )
    (1): Block(
      (mha): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): LinearLayer()
            (query): LinearLayer()
            (value): LinearLayer()
            (dropout): Dropo

In [19]:
# generate from the model

# hamlet_sample1 = "HAMLET: Not so, my lord, I am too much i' the sun.\n\nROMEO: "
# hamlet_sample2 = "HAMLET: To be, or not to be, that is the question.\n\nROMEO: "
# context = torch.tensor(tokenizer.encode(context), dtype=torch.long).unsqueeze(0).to(device)
# print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))

context = torch.zeros((1, 1), dtype=torch.long, device=device)
open('subword_large_sample.txt', 'w').write(tokenizer.decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

# generate with higher temperature --> more randomness
open('subword_large_sample_temp.txt', 'w').write(tokenizer.decode(m.generate(context, max_new_tokens=10000, temperature=1.5)[0].tolist()))

# hamlet-romeo
hamlet_sample2 = "HAMLET: To be, or not to be, that is the question.\n\nROMEO: "
context = torch.tensor(tokenizer.encode(hamlet_sample2), dtype=torch.long).unsqueeze(0).to(device)
open('subword_large_romeo.txt', 'w').write(tokenizer.decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

30823

In [20]:
# save loss dict
import json

with open('subword_large_results.json', 'w') as f:
    json.dump(results, f, indent=4)